## Executive Summary

**Purpose:** Calculate flux distribution in motor magnetic circuits

**What it does:** Solve magnetic equivalent circuits (MEC) using Newton-Raphson iteration.

### Why It Matters
Quick estimation of motor flux linkage and torque without FEA (replaces lengthy FEA simulations).

### Quick Start
**Inputs:** Circuit topology (branches, nodes), material models, MMF sources, winding definitions

**Outputs:** Flux in each circuit branch, flux linkage per winding, field energy

## How It Fits Into the Motor Design Workflow

**Upstream (depends on):** Central solver that uses results from 03_* and 04_material_models.ipynb

**Downstream (used by):** See notebook connections

In [1]:
#| hide
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple
import math

# Cross-notebook dependencies: import from the installed package so this
# notebook can run independently (nbdev pattern).
from emachines.mec import (
    PermeabilityModel,
    LinearPermeabilityModel,
    SplinePermeabilityModel,
    ShanesudhoffModel,
)

## Theory: Magnetic Equivalent Circuits

A Magnetic Equivalent Circuit (MEC) uses Ohm's law analogy:

- **Reluctance** R [A-turns/Wb] replaces resistance
- **Flux** Φ [Wb] replaces current
- **Magnetomotive force (MMF)** F [A-turns] replaces voltage

Branch equation (linear or nonlinear):
$$F = R(\Phi) \cdot \Phi + F_s$$

where F_s is an optional permanent-magnet (Norton) flux source.

### Reluctance for Nonlinear Materials

For a magnetic core with length ℓ, cross-section A, and permeability μ(B):

$$R = \frac{\ell}{\mu(B) \cdot A}, \quad B = \frac{\Phi}{A}$$

As flux increases, B increases and μ(B) may decrease (saturation), causing R to increase.

### Newton-Raphson Linearization

For each nonlinear branch, the MEC solver builds an effective (incremental) reluctance and source:

$$R_{\text{eff}} = R_0 \cdot (1 - S_0)$$
$$F_{\text{eff}} = F_s - R_0 \cdot (S_0 \cdot \Phi_0 - \Phi_{\text{source}})$$

where
- $R_0 = \ell / (\mu(B_0) \cdot A)$ — operating-point reluctance
- $S_0 = \frac{d\mu/dB \cdot B_0}{\mu}$ — saturation factor
- $B_0 = \Phi_0 / A$ — operating-point flux density

In [2]:
#| export
class BranchType:
    """Branch type identifiers for the MEC."""
    MESH       = "mesh"
    RELUCTANCE = "reluctance"
    NODAL      = "nodal"

In [3]:
#| hide
@dataclass
class _Branch:
    """Internal representation of a single MEC branch."""
    branch_id:    int
    btype:        str
    mesh_id:      Optional[int] = None
    mmf:          float = 0.0
    length:       float = 0.0
    area:         float = 0.0
    model:        Optional[Any] = None
    phi_source:   float = 0.0
    meshes:       List[int] = field(default_factory=list)
    orientations: List[int] = field(default_factory=list)
    permeance:    Optional[float] = None
    node_from:    Optional[int] = None
    node_to:      Optional[int] = None

@dataclass
class _Winding:
    """Internal descriptor of a multi-turn winding."""
    winding_id:   Any
    branch_turns: Dict[int, float] = field(default_factory=dict)

## MEC Solver Implementation

In [ ]:
#| export
MU0 = 4e-7 * math.pi

class MEC:
    """
    Magnetic Equivalent Circuit builder and Newton-Raphson solver.

    Parameters
    ----------
    phi_base : float, optional
        Base flux [Wb] for per-unit scaling (Horvath 2018).
        Typically ``B_base * A_base`` where ``B_base ≈ 1.6 T``.
        If *None*, a heuristic scale is used instead.
    F_base : float, optional
        Base MMF [A-turns] for per-unit scaling.
        Must be supplied together with *phi_base* or not at all.
    rtol, atol : float
        Newton-Raphson convergence tolerances on ‖ΔΦ‖:
        ``‖ΔΦ‖ ≤ rtol·½‖Φₖ + Φₖ₋₁‖ + atol``
    max_iter : int
        Maximum Newton-Raphson iterations.
    """

    def __init__(
        self,
        phi_base: Optional[float] = None,
        F_base: Optional[float] = None,
        rtol: float = 1e-3,
        atol: float = 1e-6,
        max_iter: int = 20,
    ) -> None:
        if (phi_base is None) != (F_base is None):
            raise ValueError("phi_base and F_base must both be specified or both be None.")

        self.phi_base = phi_base
        self.F_base = F_base
        self.rtol = rtol
        self.atol = atol
        self.max_iter = max_iter

        self._branches: Dict[int, _Branch] = {}
        self._windings: Dict[Any, _Winding] = {}
        self._next_branch_id = 0
        self._mesh_count = 0
        self._node_count = 0

In [ ]:
#| export
class MEC(MEC):  # Continue the class

    def add_mesh_branch(
        self,
        mmf: float = 0.0,
    ) -> int:
        """Add a mesh (loop current) branch.

        Parameters
        ----------
        mmf : float
            Magnetomotive force source [A-turns].

        Returns
        -------
        branch_id : int
            Unique identifier for this branch.
        """
        bid = self._next_branch_id
        self._branches[bid] = _Branch(
            branch_id=bid,
            btype=BranchType.MESH,
            mesh_id=self._mesh_count,
            mmf=mmf,
        )
        self._next_branch_id += 1
        self._mesh_count += 1
        return bid

    def add_reluctance_branch(
        self,
        length: float,
        area: float,
        model: Any,
        meshes: Sequence[int] = (),
        orientations: Sequence[int] = (),
        mmf: float = 0.0,
        phi_source: float = 0.0,
    ) -> int:
        """Add a reluctance (nonlinear magnetic) branch.

        Parameters
        ----------
        length : float
            Core length [m].
        area : float
            Cross-sectional area [m²].
        model : PermeabilityModel
            Permeability model (must have mu() and dmu_dB() methods).
        meshes : sequence of int
            Mesh IDs that cross this branch (±1).
        orientations : sequence of int
            +1 or -1 for each mesh (orientation relative to branch flux).
        mmf : float
            Magnetomotive force source [A-turns].
        phi_source : float
            PM (Norton equivalent) flux source [Wb].

        Returns
        -------
        branch_id : int
        """
        bid = self._next_branch_id
        self._branches[bid] = _Branch(
            branch_id=bid,
            btype=BranchType.RELUCTANCE,
            length=length,
            area=area,
            model=model,
            mmf=mmf,
            phi_source=phi_source,
            meshes=list(meshes),
            orientations=list(orientations),
        )
        self._next_branch_id += 1
        return bid

    def add_nodal_branch(
        self,
        permeance: float,
        node_from: int,
        node_to: int,
        mmf: float = 0.0,
    ) -> int:
        """Add a nodal (linear permeance) branch.

        Parameters
        ----------
        permeance : float
            Fixed permeance P = A/μℓ [Wb/A-turns].
        node_from : int
            Starting magnetic node.
        node_to : int
            Ending magnetic node.
        mmf : float
            Magnetomotive force source.

        Returns
        -------
        branch_id : int
        """
        bid = self._next_branch_id
        self._branches[bid] = _Branch(
            branch_id=bid,
            btype=BranchType.NODAL,
            node_from=node_from,
            node_to=node_to,
            permeance=permeance,
            mmf=mmf,
        )
        self._node_count = max(self._node_count, node_from + 1, node_to + 1)
        self._next_branch_id += 1
        return bid

In [ ]:
#| export
class MEC(MEC):  # Continue the class

    def add_winding(
        self,
        winding_id: Any,
        branch_turns: Dict[int, float],
    ) -> None:
        """Register a winding for flux linkage calculation.

        Parameters
        ----------
        winding_id : Any
            Unique identifier for this winding.
        branch_turns : dict
            Mapping {branch_id: signed_turn_count}.
            Positive = aligned with branch flux direction.
        """
        self._windings[winding_id] = _Winding(
            winding_id=winding_id,
            branch_turns=dict(branch_turns),
        )

In [ ]:
#| export
class MEC(MEC):  # Continue the class

    def solve(self) -> Any:  # Returns MECSolution
        """Solve the MEC using Newton-Raphson iteration.

        Returns
        -------
        solution : MECSolution
            Solution object with flux, MMF, field energy, and convergence info.
        """
        # Separate branches by type
        mesh_b      = [b for b in self._branches.values() if b.btype == BranchType.MESH]
        reluctance_b = [b for b in self._branches.values() if b.btype == BranchType.RELUCTANCE]

        # State vector: one entry per mesh variable
        n_mesh = len(mesh_b)
        n_nodal = self._node_count
        n_vars = n_mesh + n_nodal

        # Map branch_id -> column index in Phi
        mesh_idx = {b.branch_id: i for i, b in enumerate(mesh_b)}

        Phi = np.zeros(n_vars)
        converged = False
        residual  = float('inf')
        n_iter    = 0

        for n_iter in range(self.max_iter):
            J     = np.zeros((n_vars, n_vars))
            F_vec = np.zeros(n_vars)
            R_eff_map = {}
            Fs_eff_map = {}

            # Build one KVL equation per mesh
            for i, mesh_branch in enumerate(mesh_b):
                mmf_sum = mesh_branch.mmf  # source MMF for this loop

                for r in reluctance_b:
                    if mesh_branch.branch_id not in r.meshes:
                        continue
                    idx_in_r  = r.meshes.index(mesh_branch.branch_id)
                    orientation = r.orientations[idx_in_r]

                    phi_b = self._branch_flux(r, Phi, mesh_idx)
                    B     = phi_b / r.area if r.area > 0 else 0.0
                    mu    = r.model.mu(B)
                    dmu_dB = r.model.dmu_dB(B)

                    R0    = r.length / (mu * r.area) if mu > 0 and r.area > 0 else float('inf')
                    S0    = (dmu_dB * B / mu) if mu > 0 else 0.0
                    R_eff = R0 * (1.0 - S0) if R0 < float('inf') else float('inf')
                    Fs_eff = r.mmf - R0 * (S0 * phi_b - r.phi_source)

                    R_eff_map[r.branch_id]  = R_eff
                    Fs_eff_map[r.branch_id] = Fs_eff

                    # KVL residual: f_i = MMF_source - (R_eff*phi - Fs_eff)
                    mmf_sum -= R_eff * phi_b - Fs_eff

                    # Jacobian: df_i / d(Phi_mesh_i)
                    if R_eff < float('inf'):
                        J[i, i] -= orientation * R_eff

                F_vec[i] = mmf_sum

            # Solve J * dPhi = F  (Newton step)
            try:
                dPhi = np.linalg.solve(J + np.eye(n_vars) * 1e-12, F_vec)
            except np.linalg.LinAlgError:
                dPhi = np.linalg.pinv(J) @ F_vec

            Phi_old = Phi.copy()
            Phi    -= dPhi

            norm_Phi = 0.5 * (np.linalg.norm(Phi) + np.linalg.norm(Phi_old))
            residual = np.linalg.norm(dPhi)
            if residual <= self.rtol * norm_Phi + self.atol:
                converged = True
                break

        # ── Post-processing ──────────────────────────────────────────────────
        phi_mesh     = {b.branch_id: Phi[mesh_idx[b.branch_id]] for b in mesh_b}
        phi_branches = {}
        mmf_branches = {}

        for b in mesh_b:
            phi_branches[b.branch_id] = phi_mesh[b.branch_id]
            mmf_branches[b.branch_id] = b.mmf

        for b in reluctance_b:
            phi_b = self._branch_flux(b, Phi, mesh_idx)
            phi_branches[b.branch_id] = phi_b
            R_eff  = R_eff_map.get(b.branch_id, 0.0)
            Fs_eff = Fs_eff_map.get(b.branch_id, b.mmf)
            mmf_branches[b.branch_id] = R_eff * phi_b - Fs_eff

        flux_linkages = {
            wid: sum(N * phi_branches.get(bid, 0.0)
                     for bid, N in w.branch_turns.items())
            for wid, w in self._windings.items()
        }

        Wf = sum(
            0.5 * R_eff_map.get(b.branch_id, 0.0) * phi_branches[b.branch_id] ** 2
            for b in reluctance_b
        )

        try:
            from .result import MECSolution         # generated module context
        except ImportError:
            from emachines.mec import MECSolution   # interactive notebook context
        return MECSolution(
            phi_mesh=phi_mesh,
            phi_nodal={},
            phi_branches=phi_branches,
            mmf_branches=mmf_branches,
            flux_linkages=flux_linkages,
            converged=converged,
            n_iterations=n_iter + 1,
            residual=residual,
            field_energy=Wf,
            phi_base=self.phi_base,
            F_base=self.F_base,
        )

    def _branch_flux(self, branch, Phi, mesh_idx) -> float:
        """Return net flux through *branch* from current mesh variables."""
        return sum(
            o * Phi[mesh_idx[m]]
            for m, o in zip(branch.meshes, branch.orientations)
            if m in mesh_idx
        )

## Example: Simple Series Reluctance Circuit

In [8]:
# Import the material model
# LinearPermeabilityModel is imported in the setup cell above.
# (from emachines.mec import LinearPermeabilityModel)

# Create a simple MEC with a single mesh and reluctance
mec = MEC()

# Add a mesh with 100 A-turns source
mesh1 = mec.add_mesh_branch(mmf=100.0)

# Add an air-gap reluctance (constant permeability)
air_model = LinearPermeabilityModel(mu_r=1.0)  # Free space
reluctance1 = mec.add_reluctance_branch(
    length=0.001,  # 1 mm gap
    area=0.01,     # 100 mm² area
    model=air_model,
    meshes=[mesh1],
    orientations=[1],
)

# Solve
solution = mec.solve()
print(f"Air-gap flux: {solution.flux(reluctance1):.6e} Wb")
print(f"Converged: {solution.converged}")
print(f"Iterations: {solution.n_iterations}")

Air-gap flux: 1.256637e-03 Wb
Converged: True
Iterations: 2


## Tests

MEC solver validated against analytic solutions (series, parallel, EI-core) and material model numerics.

In [ ]:
#| hide
import math, numpy as np
from emachines.mec import MEC, ShanesudhoffModel, SplinePermeabilityModel
MU0 = 4e-7 * 3.141592653589793

# ═══ 1. Linear series circuit ══════════════════════════════════════════════════
F, R1, R2 = 1000.0, 1e6, 2e6
P1, P2 = 1/R1, 1/R2
Phi_ex = F / (R1 + R2)

mec = MEC()
mec.add_mesh_branch(branch=1, mesh=1, mmf=F)
mec.add_linear_branch(branch=2, permeance=P1, meshes=[1], orientations=[1])
mec.add_linear_branch(branch=3, permeance=P2, meshes=[1], orientations=[1])
sol = mec.solve()
assert sol.converged
assert math.isclose(sol.flux(2), Phi_ex, rel_tol=1e-8), f"branch2 flux mismatch"
assert math.isclose(sol.phi_mesh[1], Phi_ex, rel_tol=1e-8)
drop1 = sol.flux(2) / P1
drop2 = sol.flux(3) / P2
assert math.isclose(drop1 + drop2, F, rel_tol=1e-6), "KVL violated"
Wf_ex = 0.5 * (Phi_ex**2/P1 + Phi_ex**2/P2)
assert math.isclose(sol.field_energy, Wf_ex, rel_tol=1e-6)
print("✓ Linear series circuit")

# ═══ 2. Linear parallel circuit (two meshes) ══════════════════════════════════
F1, F2 = 1000.0, 500.0
Rl, Rr, Rc = 1e6, 1e6, 0.5e6
A_mat = np.array([[Rl+Rc, -Rc], [-Rc, Rr+Rc]])
b_mat = np.array([F1, F2])
phi1_ex, phi2_ex = np.linalg.solve(A_mat, b_mat)

mec2 = MEC()
mec2.add_mesh_branch(branch=1, mesh=1, mmf=F1)
mec2.add_mesh_branch(branch=2, mesh=2, mmf=F2)
mec2.add_linear_branch(branch=3, permeance=1/Rl, meshes=[1], orientations=[1])
mec2.add_linear_branch(branch=4, permeance=1/Rr, meshes=[2], orientations=[1])
mec2.add_linear_branch(branch=5, permeance=1/Rc, meshes=[1,2], orientations=[1,-1])
sol2 = mec2.solve()
assert math.isclose(sol2.phi_mesh[1], phi1_ex, rel_tol=1e-7)
assert math.isclose(sol2.phi_mesh[2], phi2_ex, rel_tol=1e-7)
assert math.isclose(sol2.flux(5), phi1_ex - phi2_ex, rel_tol=1e-7)
print("✓ Linear parallel circuit")


In [ ]:
#| hide
import math, numpy as np
from emachines.mec import MEC, ShanesudhoffModel, SplinePermeabilityModel
MU0 = 4e-7 * 3.141592653589793

# ═══ 3. ShanesudhoffModel ══════════════════════════════════════════════════════
model = ShanesudhoffModel.ferrite_3C90()
mu_r = 22340.9259
assert math.isclose(model.mu(0.0), MU0 * mu_r, rel_tol=1e-4)
for B in [0.0, 0.01, 0.1, 0.3]:
    assert model.mu(B) > 0
assert model.mu(0.30) < model.mu(0.01), "permeability should decrease with saturation"
for B in [0.05, 0.15, 0.25]:
    assert math.isclose(model.mu(B),      model.mu(-B),      rel_tol=1e-10)
    assert math.isclose(model.dmu_dB(B), -model.dmu_dB(-B),  rel_tol=1e-10)
B0, h = 0.1, 1e-7
dmu_fd = (model.mu(B0+h) - model.mu(B0-h)) / (2*h)
assert math.isclose(model.dmu_dB(B0), dmu_fd, rel_tol=1e-4)
m2 = ShanesudhoffModel(mu_r=1000.0, a=[1e-6], b=[1e6], gamma=[2.0])
assert math.isclose(m2.mu(0.0), MU0*1000.0, rel_tol=1e-2)
print("✓ ShanesudhoffModel")

# ═══ 4. SplinePermeabilityModel ════════════════════════════════════════════════
H_d = [0, 100, 500, 2000, 5000]
B_d = [0, 1.0, 1.4,  1.6,  1.8]
m = SplinePermeabilityModel(H_d, B_d)
mu_ex = MU0 * H_d[1] / B_d[1]
assert math.isclose(m.mu(0.0), mu_ex, rel_tol=1e-6)
for B in np.linspace(0, 1.8, 30):
    assert m.mu(B) > 0
for B in [0.2, 0.8, 1.5]:
    assert math.isclose(m.mu(B),      m.mu(-B),      rel_tol=1e-10)
    assert math.isclose(m.dmu_dB(B), -m.dmu_dB(-B),  rel_tol=1e-10)
B0, h = 0.9, 1e-7
dmu_fd = (m.mu(B0+h) - m.mu(B0-h)) / (2*h)
assert math.isclose(m.dmu_dB(B0), dmu_fd, rel_tol=1e-4)
mu_at_max = m.mu(B_d[-1])
assert math.isclose(m.mu(100.0), mu_at_max, rel_tol=1e-10)
try:
    SplinePermeabilityModel([1,2,3], [0,1,2])
    assert False, "should raise ValueError"
except ValueError:
    pass
print("✓ SplinePermeabilityModel")


In [ ]:
#| hide
import math, numpy as np
from emachines.mec import MEC, ShanesudhoffModel, SplinePermeabilityModel
MU0 = 4e-7 * 3.141592653589793

# ═══ 5. Nonlinear series circuit ════════════════════════════════════════════════
F_nl, l_nl, A_nl = 500.0, 0.05, 1e-4
model = ShanesudhoffModel.ferrite_3C90()

mec_nl = MEC()
mec_nl.add_mesh_branch(branch=1, mesh=1, mmf=F_nl)
mec_nl.add_nonlinear_branch(branch=2, length=l_nl, area=A_nl,
                             model=model, meshes=[1], orientations=[1])
sol_nl = mec_nl.solve()
assert sol_nl.converged, "nonlinear did not converge"
assert sol_nl.n_iterations < 50

Phi_nl = sol_nl.flux(2)
B_nl = Phi_nl / A_nl
H_nl = F_nl / l_nl
mu_nl = model.mu(B_nl)
assert math.isclose(B_nl, mu_nl * H_nl, rel_tol=1e-4), "B-H self-consistency failed"
print("✓ Nonlinear series circuit")

# ═══ 6. EI-core (parametric, linear) ══════════════════════════════════════════
N_ei, g_ei, l_c, A_ei, mu_r_eff = 100, 1e-3, 0.12, (16e-3)**2, 1000.0
for I in [0.01, 0.1, 0.5]:
    F_ei = N_ei * I
    P_core = mu_r_eff * MU0 * A_ei / l_c
    P_gap  = MU0 * A_ei / g_ei
    mec_ei = MEC()
    mec_ei.add_mesh_branch(branch=1, mesh=1, mmf=F_ei)
    mec_ei.add_linear_branch(branch=2, permeance=P_core, meshes=[1], orientations=[1])
    mec_ei.add_linear_branch(branch=3, permeance=P_gap,  meshes=[1], orientations=[1])
    sol_ei = mec_ei.solve()
    Rc = l_c / (mu_r_eff * MU0 * A_ei)
    Rg = g_ei / (MU0 * A_ei)
    Phi_ei = F_ei / (Rc + Rg)
    assert sol_ei.converged
    assert math.isclose(sol_ei.flux(2), Phi_ei, rel_tol=1e-7), f"I={I}: flux mismatch"

P_c = mu_r_eff*MU0*A_ei/l_c
P_g = MU0*A_ei/g_ei

def _ei(current):
    m = MEC(); m.add_mesh_branch(1,mesh=1,mmf=N_ei*current)
    m.add_linear_branch(2,permeance=P_c,meshes=[1],orientations=[1])
    m.add_linear_branch(3,permeance=P_g,meshes=[1],orientations=[1])
    return m.solve()

s1, s2 = _ei(0.1), _ei(0.2)
assert math.isclose(s2.flux(2), 2*s1.flux(2), rel_tol=1e-7), "linearity check failed"
assert s1.field_energy > 0
print("✓ EI-core (parametric)")


In [ ]:
#| hide
import math, numpy as np
from emachines.mec import MEC, ShanesudhoffModel, SplinePermeabilityModel
MU0 = 4e-7 * 3.141592653589793

# ═══ 7. MECSolution API ═════════════════════════════════════════════════════════
mec_api = MEC()
mec_api.add_mesh_branch(branch=10, mesh=1, mmf=1000.0)
mec_api.add_linear_branch(branch=20, permeance=1e-6, meshes=[1], orientations=[1])
sol_api = mec_api.solve()

assert math.isclose(sol_api.flux(20),  sol_api.phi_branches[20],  rel_tol=1e-12)
assert math.isclose(sol_api.mmf(20),   sol_api.mmf_branches[20],  rel_tol=1e-12)
assert "converged" in repr(sol_api)
assert "heuristic" in repr(sol_api)
try: sol_api.flux(999); assert False
except KeyError: pass

mec_pua = MEC(phi_base=1e-4, F_base=1e3)
mec_pua.add_mesh_branch(branch=10, mesh=1, mmf=1000.0)
mec_pua.add_linear_branch(branch=20, permeance=1e-6, meshes=[1], orientations=[1])
assert "pu" in repr(mec_pua.solve())
print("✓ MECSolution API")

# ═══ 8. Per-unit scaling ════════════════════════════════════════════════════════
F_pu, P_pu = 1000.0, 1e-6
Phi_pu = F_pu * P_pu
phi_b, F_b = Phi_pu * 1.1, F_pu * 1.5

def _simple(phi_base=None, F_base=None):
    kw = dict(phi_base=phi_base, F_base=F_base) if phi_base else {}
    m = MEC(**kw)
    m.add_mesh_branch(1, mesh=1, mmf=F_pu)
    m.add_linear_branch(2, permeance=P_pu, meshes=[1], orientations=[1])
    return m.solve()

assert math.isclose(_simple(phi_b, F_b).flux(2), _simple().flux(2), rel_tol=1e-8)
assert math.isclose(_simple(phi_b, F_b).phi_base, phi_b)
assert _simple().phi_base is None

try: MEC(phi_base=1e-4, F_base=None); assert False
except ValueError: pass
try: MEC(phi_base=None, F_base=1e3); assert False
except ValueError: pass

l_pnl, A_pnl = 0.05, 1e-4
B_est = 0.2
phi_bb = B_est * A_pnl
F_bb   = phi_bb / (MU0 * A_pnl / l_pnl)
mec_pnl = MEC(phi_base=phi_bb, F_base=F_bb)
mec_pnl.add_mesh_branch(1, mesh=1, mmf=500.0)
mec_pnl.add_nonlinear_branch(2, length=l_pnl, area=A_pnl,
    model=ShanesudhoffModel.ferrite_3C90(), meshes=[1], orientations=[1])
sol_pnl = mec_pnl.solve()
assert sol_pnl.converged and sol_pnl.n_iterations < 50
print("✓ Per-unit scaling")

# ═══ 9. Flux linkage (windings) ════════════════════════════════════════════════
F_w, R1_w, R2_w = 1000.0, 1e6, 2e6
Phi_w = F_w / (R1_w + R2_w)
N_w = 100

def _mec_w(branch_turns=None):
    m = MEC()
    m.add_mesh_branch(1, mesh=1, mmf=F_w)
    m.add_linear_branch(2, permeance=1/R1_w, meshes=[1], orientations=[1])
    m.add_linear_branch(3, permeance=1/R2_w, meshes=[1], orientations=[1])
    if branch_turns: m.add_winding("coil", branch_turns)
    return m.solve()

s = _mec_w({2: N_w})
assert "coil" in s.flux_linkages
assert math.isclose(s.flux_linkage("coil"), N_w*Phi_w, rel_tol=1e-7)
assert math.isclose(_mec_w({2: N_w, 3: N_w}).flux_linkage("coil"), 2*N_w*Phi_w, rel_tol=1e-7)

mec_opp = MEC()
mec_opp.add_mesh_branch(1, mesh=1, mmf=1000.0)
mec_opp.add_linear_branch(2, permeance=1e-6, meshes=[1], orientations=[1])
mec_opp.add_linear_branch(3, permeance=1e-6, meshes=[1], orientations=[1])
mec_opp.add_winding("coil", {2: +100, 3: -100})
assert math.isclose(mec_opp.solve().flux_linkage("coil"), 0.0, abs_tol=1e-15)

mec_mw = MEC()
mec_mw.add_mesh_branch(1, mesh=1, mmf=F_w)
mec_mw.add_linear_branch(2, permeance=1/R1_w, meshes=[1], orientations=[1])
mec_mw.add_linear_branch(3, permeance=1/R2_w, meshes=[1], orientations=[1])
mec_mw.add_winding("primary", {2: 100})
mec_mw.add_winding("secondary", {3: 50})
s_mw = mec_mw.solve()
assert math.isclose(s_mw.flux_linkage("primary"),  100*Phi_w, rel_tol=1e-7)
assert math.isclose(s_mw.flux_linkage("secondary"), 50*Phi_w, rel_tol=1e-7)

assert _mec_w().flux_linkages == {}
try: _mec_w({2: N_w}).flux_linkage("nonexistent"); assert False
except KeyError: pass
s_chk = _mec_w({2: N_w})
assert s_chk.flux_linkage("coil") == s_chk.flux_linkages["coil"]
print("✓ Flux linkage / windings")

# ═══ 10. Scaling consistency ════════════════════════════════════════════════════
P_sc = MU0 * 4e-4 / 1e-3
mec_sc1 = MEC()
mec_sc1.add_mesh_branch(1, mesh=1, mmf=800.0)
mec_sc1.add_linear_branch(2, permeance=P_sc, meshes=[1], orientations=[1])
assert mec_sc1.solve().converged

phi_bsc = 1.6 * 4e-4
F_bsc   = phi_bsc / P_sc
mec_sc2 = MEC(phi_base=phi_bsc, F_base=F_bsc)
mec_sc2.add_mesh_branch(1, mesh=1, mmf=800.0)
mec_sc2.add_linear_branch(2, permeance=P_sc, meshes=[1], orientations=[1])
assert math.isclose(mec_sc2.solve().flux(2), mec_sc1.solve().flux(2), rel_tol=1e-6)
print("✓ Scaling consistency")

print("\n✓ All MEC solver tests passed")
